# Met Eyes Experiments

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
import json
import requests

from os import listdir, path
from PIL import Image as PImage
from time import sleep

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

In [ ]:
MET_URL = "https://collectionapi.metmuseum.org/public/collection/v1"

SEARCH_DEPARTMENT_IDS = []
SEARCH_DEPARTMENTS = ["Robert Lehman", "Armor"]
SEARCH_MEDIUMS = ["Paintings", "Drawings"][:1]

### Helper for combining jsons

In [ ]:
def export_combined_jsons(input_json_dir, output_json_dir, filename):
  json_files = sorted(f for f in listdir(input_json_dir) if f.endswith("json"))
  combined_data = []
  for jf in json_files:
    with open(f"{input_json_dir}/{jf}", "r") as ifp:
      combined_data.append(json.load(ifp))
  with open(f"{output_json_dir}/{filename}.json", "w") as ofp:
    json.dump({ filename : combined_data }, ofp)

### Get Department IDs

In [ ]:
dept_response = requests.get(f"{MET_URL}/departments")
dept_data = dept_response.json()["departments"]

dept_name2id = { d["displayName"] : d["departmentId"] for d in dept_data }

for sdpt in SEARCH_DEPARTMENTS:
  for dname,did in dept_name2id.items():
    if sdpt.lower() in dname.lower():
      SEARCH_DEPARTMENT_IDS.append(did)

### Get Object IDs

In [ ]:
obj_ids = []

for dpt_query in SEARCH_DEPARTMENT_IDS:
  for medium_query in SEARCH_MEDIUMS:
    collection_response = requests.get(f"{MET_URL}/search?medium={medium_query}&departmentId={dpt_query}&q=*")
    query_obj_ids = set(collection_response.json()["objectIDs"])
    obj_ids += list(query_obj_ids)

len(obj_ids)

### Get Object Metadata

In [ ]:
obj_fields = ["objectID", "objectName", "title", "primaryImage", "primaryImageSmall", "artistRole", "artistDisplayName"]
obj_files = sorted(f for f in listdir(f"{JSON_DIR}/objects") if f.endswith("json"))

for oid in obj_ids:
  obj_json_path = f"{JSON_DIR}/objects/{oid}.json"
  if f"{oid}.json" in obj_files:
    continue

  obj_response = requests.get(f"{MET_URL}/objects/{oid}")
  obj_data = obj_response.json()
  obj_filtered_data = { f: obj_data[f] for f in obj_fields }

  obj_json_path = f"{JSON_DIR}/objects/{oid}.json"
  with open(obj_json_path, "w") as ofp:
    json.dump(obj_filtered_data, ofp)
  sleep(0.333)

### Export Combined Object Metadata

In [ ]:
export_combined_jsons(f"{JSON_DIR}/objects", f"{JSON_DIR}", "objects")

### Get Images

In [ ]:
with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]

for obj in obj_data:
  img_url = obj["primaryImage"]
  if not (img_url and len(img_url) > 0):
    continue

  oid = obj["objectID"]
  img_hd_path = f"{IMG_DIR}/hd/{oid}.jpg"
  img_900_path = img_hd_path.replace("/hd/", "/900/")
  img_500_path = img_hd_path.replace("/hd/", "/500/")
  if path.isfile(img_hd_path) and path.isfile(img_900_path) and path.isfile(img_500_path):
    continue

  img_response = requests.get(img_url, stream=True)
  img = PImage.open(img_response.raw)

  img.thumbnail((2048, 2048))
  if not path.isfile(img_hd_path):
    img.save(img_hd_path)

  img.thumbnail((900, 900))
  if not path.isfile(img_900_path):
    img.save(img_900_path)

  img.thumbnail((500, 500))
  if not path.isfile(img_500_path):
    img.save(img_500_path)

  sleep(0.333)

### Analyze Faces

In [ ]:
# TODO: OpenCV
# TODO: https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/index
# TODO: https://huggingface.co/kartiknarayan/facexformer
# TODO: https://huggingface.co/qualcomm/Facial-Landmark-Detection